---
layout: post
title: Open Coding Society - Lesson Gamify
description: Start interactive experience by pressing "Play".
permalink: /Gamify-Lesson
codemirror: true
hide: true
toc: false
---

# Enemy Collision, Combat & Death System

> **Course:** MarketPirateGame Engine · **Topic:** Collision & Combat · **Level:** Intermediate JavaScript

---

## Overview

The game uses a **proximity-based collision system** rather than pixel-perfect hitbox collision. Instead of checking whether two sprites overlap on a canvas, it measures the *logical distance* between the player's position and each enemy's position every frame. When that distance falls below a threshold, the game treats it as a "collision" and prompts the player to fight.

| Concept | Value | Notes |
|:---|:---:|:---|
| Detection method | Distance | Euclidean — `Math.hypot(dx, dy)` |
| Collision radius | `120px` | Logical pixels, not screen pixels |
| Battle trigger | `E` key | Only fires when an enemy is nearby |
| Respawn interval | `25s` | Cleans dead enemies, spawns new ones |


---

## 1 · How Enemy Position Works

Each enemy is a `WorldEnemy` instance. When spawned, it is given a **logical X/Y position** relative to the game container — not the screen viewport.

```js
class WorldEnemy {
  constructor(enemyType, logicalX, logicalY, container) {
    this.logicalX = logicalX;
    this.logicalY = logicalY;
    this._container = container;
    // ...
  }
}
```

These logical coordinates are then translated to **screen-space pixels** using the container's bounding rectangle:

```js
_syncPosition() {
  const rect = this._container.getBoundingClientRect();
  this.el.style.left = (rect.left + this.logicalX) + 'px';
  this.el.style.top  = (rect.top  + this.logicalY) + 'px';
}
```

> **Why logical coords?**
> The game world scrolls and resizes. Logical positions stay relative to the container so enemies don't drift when the browser window changes. `_syncPosition()` is called every `resize()` to keep markers correct.

---

## 2 · Player Position Extraction

The player's logical position is read from the `Player` game object each frame inside `update()`.

```js
_playerLogicalPos() {
  const p = this.gameEnv.gameObjects?.find(o => o instanceof Player);
  if (!p?.position) return null;

  return {
    x: p.position.x + (p.width  || 0) * 0.5,  // horizontal center
    y: p.position.y + (p.height || 0) * 0.8,  // near the player's feet
  };
}
```

> **Foot offset (0.8):** The Y position is anchored near the player's feet, not their center. This makes proximity feel natural — the player "walks up to" an enemy rather than floating into them.

---

## 3 · Proximity Collision Detection

Every frame, `_findNearbyEnemy()` loops over all living enemies and computes the **Euclidean distance** to the player.

```js
_findNearbyEnemy(threshold = 120) {
  const pos = this._playerLogicalPos();
  if (!pos) return null;

  let closest = null;
  let best    = threshold;

  for (const e of this._worldEnemies) {
    if (e.defeated) continue;                            // skip dead enemies

    const d = Math.hypot(pos.x - e.logicalX, pos.y - e.logicalY);

    if (d < best) {
      best    = d;
      closest = e;
    }
  }
  return closest; // null if no enemy is within 120px
}
```

**Key points:**

- `threshold = 120` — detection radius in logical pixels
- `Math.hypot(dx, dy)` — true 2D distance (Pythagorean theorem)
- Returns only the **single nearest** enemy — multiple simultaneous battles aren't possible
- `e.defeated` check — dead enemies are invisible to the system

If the returned value is not `null`, the HUD displays a prompt and pressing `E` triggers the battle.

---

## 4 · The `update()` Loop

`update()` runs every game tick and ties everything together.

```js
update() {
  if (!this._gameStarted) return;

  const player = this.gameEnv.gameObjects.find(o => o instanceof Player);
  if (!player) return;

  // Don't check collisions while a shop or battle is open
  if (this._open || this._battleOpen || this._specOpen) {
    this._setHint('');
    this._worldEnemies.forEach(e => e.hideHint());
    return;
  }

  const nearby = this._findNearbyEnemy(120);
  this._nearbyEnemy = nearby;

  this._worldEnemies.forEach(e => e.hideHint());

  if (nearby) {
    nearby.showHint('Press E to fight');
    this._setHint(`⚔ ${nearby.type.name} — Lv.${nearby.type.level} · Press E to fight`);
  }
}
```

**Guards in this loop:**

| Condition | Effect |
|:---|:---|
| `_gameStarted === false` | Entire loop is skipped (still on menu) |
| `_open \|\| _battleOpen \|\| _specOpen` | Collision suspended — prevents double-triggering |

---

## 5 · Triggering the Battle

When the player presses `E`, the key handler checks `_nearbyEnemy`:

```js
this._keyHandler = (e) => {
  if (e.key !== 'e' && e.key !== 'E') return;
  if (this._battleOpen || this._open || this._specOpen) return;

  if (this._nearbyEnemy) {
    this._startBattle(this._nearbyEnemy);  // collision → fight!
    return;
  }
  // otherwise check shop zones...
};
```

`_startBattle()` opens the `BattleUI` modal and passes through the player's current stats:

```js
_startBattle(worldEnemy) {
  if (this._battleOpen) return;
  this._battleOpen = true;
  this._nearbyEnemy = null;

  this._battleUI = new BattleUI(
    worldEnemy.type,
    (rubyReward, wasDefeated, remainingHp, xpGained) => {
      // --- battle callback ---
      this._battleOpen = false;
      this._battleUI   = null;

      if (wasDefeated) {
        this._showGameOver();
      } else {
        this._playerCurHp = Math.max(1, remainingHp);
        this._updateWorldHP();

        if (rubyReward > 0) {
          worldEnemy.markDefeated();  // ← enemy death happens here
          this._killCount++;
          this._bankedRubies += rubyReward;
        }
        if (xpGained > 0) this._gainXp(xpGained);
      }
    },
    this._playerCurHp,
    this._playerLevel,
    this._playerAtk,
    this._playerDef,
    this._playerMaxHp,
  );
}
```

---

## 6 · Enemy Death — `markDefeated()`

When the player wins a battle and earns rubies, `worldEnemy.markDefeated()` is called:

```js
markDefeated() {
  this.defeated = true;                     // 1. flag — skips future collision checks
  this.hideHint();                          // remove tooltip bubble
  this.el.classList.add('defeated');        // 2. trigger CSS fade-out
  setTimeout(() => this.el.remove(), 400); // 3. DOM cleanup after animation
}
```

The CSS powering the visual death:

```css
/* transition: opacity .3s is defined on .world-enemy */
.world-enemy.defeated {
  opacity: 0;
  pointer-events: none;  /* no further mouse interaction */
}
```

**Three things happen in sequence:**

1. **`this.defeated = true`** — the flag causes all future collision loops to skip this enemy instantly
2. **`.defeated` class** — CSS transitions the enemy to `opacity: 0` over 300ms
3. **`el.remove()`** — called 400ms later, after the animation has finished

The enemy object stays in the `_worldEnemies` array briefly, but the flag makes it invisible to the system. The respawn timer cleans it out:

```js
this._respawnTimer = setInterval(() => {
  this._worldEnemies = this._worldEnemies.filter(e => !e.defeated);
  if (this._worldEnemies.length < 4) this._spawnEnemies(3);
}, 25000);
```

---

## 7 · Spawning & Respawning

Enemies spawn with a **zone exclusion guard** that keeps them away from all market zones:

```js
_spawnEnemies(n) {
  const allZones = [
    { cx: this.shopZone.x + this.shopZone.width  / 2,
      cy: this.shopZone.y + this.shopZone.height / 2 },
    ...this._specZones.map(z => ({ cx: z.cx, cy: z.cy })),
  ];

  for (let i = 0; i < n; i++) {
    let lx, ly, tries = 0;
    do {
      lx = margin + Math.random() * (width  - margin * 2);
      ly = margin + Math.random() * (height - margin * 2);
      tries++;
    } while (tries < 30 && allZones.some(z => Math.hypot(lx - z.cx, ly - z.cy) < 120));

    this._worldEnemies.push(new WorldEnemy(type, lx, ly, container));
  }
}
```

> The `do...while` retries up to **30 times** to find a position at least **120px** away from every market zone center. This prevents enemies from spawning inside shops.

---

## 8 · Full Death Lifecycle

```
Player walks near enemy
        │
        ▼
_findNearbyEnemy()  →  distance < 120px  →  returns WorldEnemy
        │
        ▼
Player presses E
        │
        ▼
_startBattle(worldEnemy)  →  BattleUI opens
        │
   [player wins]
        │
        ▼
Battle callback fires
        ├── worldEnemy.markDefeated()
        │         ├── this.defeated = true
        │         ├── CSS fade-out  (.defeated class, 300ms)
        │         └── DOM removal   (setTimeout 400ms)
        ├── _killCount++
        ├── rubies banked
        └── _gainXp(xpGained)

After 25 seconds:
  _worldEnemies.filter(e => !e.defeated)
  count < 4  →  _spawnEnemies(3)
```

---

## 9 · Student Experiments

Try these modifications to deepen your understanding:

### Change the detection radius
Increase or decrease the `120` threshold in `_findNearbyEnemy()`. What happens at `50`? At `300`?

```js
_findNearbyEnemy(threshold = 50) { ... }  // much smaller range
```

### Give enemies HP — require multiple battles
Track enemy HP and only call `markDefeated()` when it reaches zero:

```js
class WorldEnemy {
  constructor(enemyType, logicalX, logicalY, container) {
    // ...
    this.hp = enemyType.maxHp ?? 3;  // survives multiple fights
  }
}
```

### Add a death particle burst
Hook into `markDefeated()` to spawn visual effects:

```js
markDefeated() {
  this.defeated = true;
  this.hideHint();
  spawnParticles(this.logicalX, this.logicalY);  // your effect here
  this.el.classList.add('defeated');
  setTimeout(() => this.el.remove(), 400);
}
```

### Slow the respawn timer
Change `25000` to `5000` and watch how world density changes. What's the best pacing for the game feel?

---

## 10 · Quick Reference

| Concept | Where | Key Detail |
|:---|:---|:---|
| Enemy position | `WorldEnemy._syncPosition()` | Logical coords + container rect offset |
| Player position | `_playerLogicalPos()` | Feet position — 0.8 × height |
| Collision check | `_findNearbyEnemy(120)` | Euclidean distance, every frame |
| Battle trigger | `_keyHandler → _startBattle()` | E keypress within 120px only |
| Enemy death flag | `WorldEnemy.defeated` | Skips enemy in all future checks |
| Visual death | `.defeated` CSS class | `opacity: 0` + `pointer-events: none` |
| DOM cleanup | `setTimeout(el.remove, 400)` | After fade animation completes |
| Array cleanup | `_respawnTimer` filter | Every 25 seconds |
| Spawn guard | `do...while` with zone check | 120px clear of all market zones |




## Interactive Game Engine:

**download the files bellow**

Steps:
<br>
1. Download each file

2. Make a local folder & place the all downloaded files into the folder

3. After the Files are in ther double click on the HTML file and open it on the broswer
   a. for windows double click to open
   b. for other computers open by simpily right clicking on the html file then click "Open in Browser"

4. create a new project and paste the code below:

In [ ]:
// OCS Platformer Demo
const player = new Actor(320, 100);
player.vy = 0;
player.gravity = 0.7;

const ground = new Actor(0, 320);
ground.width = 640; ground.height = 40;
ground.color = "#238636";

player.update = function() {
    if (Engine.keys.KeyD) this.x += 5;
    if (Engine.keys.KeyA) this.x -= 5;

    this.vy += this.gravity;
    this.y += this.vy;

    if (this.y + this.height > ground.y) {
        this.y = ground.y - this.height;
        this.vy = 0;
    }

    if (Engine.keys.Space && this.vy === 0) this.vy = -12;
};

Engine.objects.push(ground, player);

## Manual Download:

<a href="index.html" download>⬇ Download index.html</a>

<a href="style.css" download>⬇ Download style.css</a>

<a href="engine.js" download>⬇ Download engine.js</a>

<br>
Quick Download:
<br>
<div style="text-align:center; margin: 40px 0;">

<a class="download-btn" 
   href="https://csse-2.github.io/Project-S.A.R/downloads/OCS%20Game%20Engine.zip"
   download>
   ⬇️ Download OCS Game Engine (ZIP)
</a>

</div>

<style>
.download-btn {
    background: #003c78;
    color: white;
    padding: 20px 40px;
    font-size: 24px;
    border-radius: 12px;
    text-decoration: none;
    font-weight: bold;
    display: inline-block;
    box-shadow: 0 0 20px rgba(83, 169, 255, 0.8);
    transition: all 0.25s ease;
    animation: float 2.5s ease-in-out infinite;
}

/* Hover effects */
.download-btn:hover {
    background: #005fb8;
    border-radius: 20px;
    transform: translateY(-6px) scale(1.05);
    box-shadow: 0 0 30px rgba(135, 206, 250, 1);
}

/* Bounce animation */
@keyframes float {
    0% { transform: translateY(0px); }
    50% { transform: translateY(-6px); }
    100% { transform: translateY(0px); }
}
</style>
